# Qwen-Image-Edit quality experiment (standalone, not wired into the backend)

**Purpose:** run the actual Qwen-Image-Edit model (4-bit Nunchaku quantized, so it fits a free Colab T4 GPU) on your room photo, using the *same* Economical/Mid/Premium tier prompts as the production `app/pipeline/prompts.py` (v5), so the output is directly comparable to what Cloudflare/SD1.5 currently produces.

**Read before running - real limitations, not swept under the rug:**
- This is Google Colab's **free** tier: a T4 GPU (16GB VRAM), sessions disconnect after ~90 min idle and hard-cap around ~12 hours, and GPU availability isn't guaranteed. This notebook is a **manual, one-off experiment** to compare quality - it is *not* wired into the FastAPI backend and isn't meant to run unattended as a service.
- **Runtime > Change runtime type > T4 GPU** must be selected before running any cell below, or nothing will work.
- The `nunchaku` install step (step 2) auto-detects your Python/Torch/CUDA versions and queries GitHub's releases API for a matching wheel at runtime - if it still can't find one, it raises a clear error rather than installing something wrong; browse https://github.com/nunchaku-ai/nunchaku/releases manually in that case (note: the org is `nunchaku-ai`, not `nunchaku-tech` - it was renamed and older docs/blog posts may show the old name).
- First run downloads several GB of model weights from Hugging Face - expect this to take a few minutes.

## 1. Environment check
Run this first. Confirms you actually have a GPU runtime, and prints the exact versions needed to pick a matching `nunchaku` wheel if the automatic install in step 2 fails.

In [ ]:
import sys
import torch

print(f"Python: {sys.version}")
print(f"Torch:  {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")
else:
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > T4 GPU, then re-run this cell."
    )

## 2. Install dependencies

`diffusers`/`transformers`/`accelerate`/`bitsandbytes` install cleanly from PyPI. `nunchaku` does not - it ships as prebuilt wheels per exact Torch/CUDA/Python combination, published as GitHub release assets rather than on PyPI. The next cell queries the GitHub releases API at runtime and picks the wheel matching *your actual* environment automatically, instead of a hardcoded guess (a hardcoded URL 404'd previously because Colab's torch/CUDA version didn't match what was guessed, and because the wheel filenames also embed a CUDA tag like `cu12.8` that a plain `torch2.8` guess omitted).

In [ ]:
%pip install -q -U diffusers transformers accelerate bitsandbytes pillow

In [ ]:
import re
import sys
import urllib.request
import json

import torch

py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
torch_ver = torch.__version__.split("+")[0]
torch_major_minor = ".".join(torch_ver.split(".")[:2])
cuda_ver = torch.version.cuda  # e.g. "12.8"

print(f"Detected: Python {py_tag}, Torch {torch_major_minor}, CUDA {cuda_ver}")

with urllib.request.urlopen(
    "https://api.github.com/repos/nunchaku-ai/nunchaku/releases/tags/v1.2.1"
) as resp:
    release = json.loads(resp.read())

candidates = [
    a["browser_download_url"]
    for a in release["assets"]
    if a["name"].endswith(".whl") and "linux_x86_64" in a["name"] and py_tag in a["name"]
]

# Prefer an exact CUDA+Torch match; fall back to a Torch-only match; fall back to any cp-matching wheel.
exact = [u for u in candidates if f"cu{cuda_ver}torch{torch_major_minor}" in u]
torch_only = [u for u in candidates if f"torch{torch_major_minor}" in u]

if exact:
    wheel_url = exact[0]
elif torch_only:
    wheel_url = torch_only[0]
    print(f"WARNING: no exact CUDA match, using a Torch-{torch_major_minor} wheel with a different CUDA tag.")
elif candidates:
    wheel_url = candidates[0]
    print(f"WARNING: no Torch-{torch_major_minor} match either, falling back to: {wheel_url}")
else:
    raise RuntimeError(
        f"No nunchaku v1.2.1 wheel found for Python {py_tag} at all. "
        "Browse https://github.com/nunchaku-ai/nunchaku/releases manually."
    )

print(f"Installing: {wheel_url}")
%pip install -q "$wheel_url"

### 2b. Fix a known Pillow version conflict

The `nunchaku` install above can silently downgrade Pillow to a version older than what `diffusers` needs, causing `ImportError: cannot import name '_Ink' from 'PIL._typing'` when you try to load the model in step 3. This cell forces Pillow back to the latest version.

**After running this cell, go to Runtime > Restart session, then re-run every cell from the top** (installs are fast the second time, they're cached). Python doesn't reload already-imported native extensions like Pillow's C code within the same session, so skipping the restart will keep hitting the same error even after this fix.

In [ ]:
%pip install -q -U --force-reinstall --no-deps pillow
print("Pillow upgraded. Now go to Runtime > Restart session, then re-run all cells from the top.")

## 3. Load the model

4-bit Nunchaku-quantized diffusion transformer (only ~3-4GB VRAM via per-layer offloading on GPUs under 18GB, like the free T4). **The text encoder also needs quantizing** - it's a separate 8.3B-parameter Qwen2.5-VL model that Nunchaku doesn't touch (Nunchaku only quantizes the diffusion transformer), and at full bf16 it alone needs ~16GB, which is what crashed the session ("used all available RAM") even though the transformer fit fine in VRAM - free Colab's system RAM is ~12-13GB, far below Qwen's own documented 64GB recommendation for the unquantized setup. This cell 4-bit-quantizes the text encoder too, via `bitsandbytes`.

In [ ]:
import torch
from diffusers import QwenImageEditPlusPipeline
from transformers import BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

from nunchaku import NunchakuQwenImageTransformer2DModel
from nunchaku.utils import get_gpu_memory, get_precision

RANK = 128  # higher rank = better quality, more VRAM/time; 32 is the faster alternative

# 4-bit quantize the 8.3B text encoder (confirmed class/subfolder from the repo's own
# model_index.json) - this is what crashed the free-tier session at full bf16.
text_encoder_quant_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16
)
text_encoder = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen-Image-Edit-2509",
    subfolder="text_encoder",
    quantization_config=text_encoder_quant_config,
    torch_dtype=torch.bfloat16,
)

transformer = NunchakuQwenImageTransformer2DModel.from_pretrained(
    f"nunchaku-ai/nunchaku-qwen-image-edit-2509/"
    f"svdq-{get_precision()}_r{RANK}-qwen-image-edit-2509.safetensors"
)

pipeline = QwenImageEditPlusPipeline.from_pretrained(
    "Qwen/Qwen-Image-Edit-2509",
    transformer=transformer,
    text_encoder=text_encoder,
    torch_dtype=torch.bfloat16,
)

if get_gpu_memory() > 18:
    pipeline.enable_model_cpu_offload()
else:
    # Free T4 (16GB) path: per-layer offloading, ~3-4GB VRAM.
    transformer.set_offload(True, use_pin_memory=False, num_blocks_on_gpu=1)
    pipeline._exclude_from_cpu_offload.append("transformer")
    pipeline.enable_sequential_cpu_offload()

print("Model loaded.")

## 4. Upload your room photo

In [ ]:
from google.colab import files
from PIL import Image
import io

uploaded = files.upload()
room_photo_bytes = next(iter(uploaded.values()))
room_photo = Image.open(io.BytesIO(room_photo_bytes)).convert("RGB")
room_photo

## 5. Tier prompts
Ported directly from the production `app/pipeline/prompts.py` (v5) so this is an apples-to-apples comparison against the live Cloudflare/SD1.5 pipeline - same tier methodology, same PRESERVE_STRUCTURE block, same premium structure_reminder fix, same negative-prompt exclusions. If you tune prompts.py later, re-copy the relevant constants here to keep this comparable.

In [ ]:
TIER_SPECS = {
    "economical": {
        "label": "budget renovation",
        "paint": "plain flat cream and sage-green two-tone paint, cheap and utilitarian looking",
        "flooring": "ordinary matte grey ceramic tile flooring, plain and unpolished, no shine, no gloss",
        "lighting_temp": "cool white practical lighting, 5000-6000K",
        "feature_wall": "no accent wall, no wall mouldings, bare plain walls",
        "ceiling": "plain flat white ceiling, no false ceiling, a simple ceiling fan",
        "materials": "paint only, no premium materials, no wood paneling, no marble",
        "palette": "muted cream, sage green, and grey tones",
        "density": "sparse furniture, about 80 percent of floor space left empty, uncluttered and clean",
        "decor": "one or two potted plants, simple thin plain curtains, one or two plain framed prints",
        "structure_reminder": "",
    },
    "mid": {
        "label": "mid-level renovation",
        "paint": "warm beige walls with olive-green wainscoting panel on the lower half",
        "flooring": "warm wood-look laminate flooring",
        "lighting_temp": "neutral warm lighting, 3500-4000K",
        "feature_wall": "wall mouldings and a few framed art prints",
        "ceiling": "false ceiling with a warm cove lighting strip",
        "materials": "paint, wall mouldings, wainscoting, laminate wood flooring",
        "palette": "warm beige, olive green, and light wood tones",
        "density": "balanced furniture arrangement, tidy and comfortable, not crowded",
        "decor": "an area rug, framed art prints, a few potted plants",
        "structure_reminder": "",
    },
    "premium": {
        "label": "premium luxury renovation",
        "paint": "dark wood panel and marble feature wall with brass trim accents",
        "flooring": "polished Italian marble flooring, reflective and bright",
        "lighting_temp": "warm luxury lighting, 2700-3000K, like a luxury hotel lobby",
        "feature_wall": "marble and dark wood panel wall with brass inlay",
        "ceiling": "designer multi-layer cove ceiling with warm gold-lit trim",
        "materials": "real marble, brass trim, dark wood paneling",
        "palette": "rich dark wood tones with gold and brass metallic accents",
        "density": "furniture arranged in curated symmetrical conversation zones, restrained, not overfilled",
        "decor": "floor-to-ceiling heavy fabric curtains, framed art, symmetrical furniture placement",
        "structure_reminder": "still the same original room shape and window, do not enlarge or change the space",
    },
}

PRESERVE_STRUCTURE = (
    "same room structure, same walls, same windows, same doors, same ceiling height, "
    "same camera angle and perspective as the original photo, unchanged room dimensions"
)

NEGATIVE_ADDITIONS = {
    "economical": "chandelier, crystal chandelier, pendant light, gold trim, marble, luxury, ornate, wainscoting",
    "mid": "chandelier, crystal chandelier, gold trim, marble, ornate luxury details",
    "premium": "",
}

UNIVERSAL_DAMAGE_NEGATIVE = (
    "damaged, dirty, stained, cracked walls, cracked ceiling, mold, mildew, water "
    "damage, peeling paint, debris, rubble, dust, disrepair, abandoned, derelict"
)


def build_prompt(tier: str, room_description: str | None = None) -> str:
    spec = TIER_SPECS[tier]
    tokens = [f"{spec['label']} of the same room", PRESERVE_STRUCTURE]
    if room_description:
        tokens.append(room_description)
    tokens += [spec["paint"], spec["flooring"], spec["lighting_temp"], spec["ceiling"], spec["feature_wall"]]
    if spec.get("structure_reminder"):
        tokens.append(spec["structure_reminder"])
    tokens += [spec["materials"], spec["palette"], spec["density"], spec["decor"]]
    return ", ".join(tokens)


def build_negative_prompt(tier: str) -> str:
    addition = NEGATIVE_ADDITIONS[tier]
    if addition:
        return f"{UNIVERSAL_DAMAGE_NEGATIVE}, {addition}"
    return UNIVERSAL_DAMAGE_NEGATIVE


for tier in TIER_SPECS:
    print(f"--- {tier} ---")
    print("POSITIVE:", build_prompt(tier))
    print("NEGATIVE:", build_negative_prompt(tier))
    print()

## 6. Generate all 3 tiers
Qwen-Image-Edit doesn't use img2img's `strength` knob (that's SD1.5-specific) - it's an instruction-following edit model, so structure preservation comes from the prompt itself (the same PRESERVE_STRUCTURE + structure_reminder text used above) rather than a noise-strength parameter. `true_cfg_scale` and `num_inference_steps` are the closest tunable knobs; the values below are the model authors' own defaults - lower `num_inference_steps` (e.g. 20-25) if you want faster iteration at some quality cost.

In [ ]:
results = {}

for tier in TIER_SPECS:
    print(f"Generating {tier}...")
    output = pipeline(
        image=[room_photo],
        prompt=build_prompt(tier),
        negative_prompt=build_negative_prompt(tier),
        true_cfg_scale=4.0,
        num_inference_steps=40,
    )
    results[tier] = output.images[0]
    results[tier].save(f"qwen_{tier}.png")

print("Done. Saved qwen_economical.png, qwen_mid.png, qwen_premium.png")

## 7. View side by side

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(room_photo)
axes[0].set_title("Original")
for ax, tier in zip(axes[1:], TIER_SPECS):
    ax.imshow(results[tier])
    ax.set_title(tier.capitalize())
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()